In [ ]:
import pandas as pd, numpy as np

clean = pd.read_csv('cafe_sales_clean_fixed.csv', sep='\t')
raw = pd.read_csv('dirty_cafe_sales.csv'); raw.columns = clean.columns

# 1. Flag baris yang tanggalnya diacak
ok = raw.transaction_date.fillna('').str.match(r'^\d{4}-\d{2}-\d{2}$')
df = clean.merge(raw[['transaction_id']].assign(ok=ok), on='transaction_id')
df['date_imputed'] = ~df.ok
df['transaction_date'] = pd.to_datetime(df.transaction_date)
df['month'] = df.transaction_date.dt.month

# 2-4. Dampak & perbandingan A vs B
print(df.date_imputed.sum(), df[df.date_imputed].total_spent.sum())
A = df.groupby('month').total_spent.sum()
B = df[~df.date_imputed].groupby('month').total_spent.sum()
print(A.corr(B), A.corr(B, method='spearman'))

# 5. Monte Carlo: acak ulang tanggal 1.000 kali
rng = np.random.default_rng(1)
dates = pd.date_range('2023-01-01', '2023-12-31')
imp = df[df.date_imputed]
sims = []
for _ in range(1000):
    m = pd.DatetimeIndex(rng.choice(dates, len(imp))).month
    s = imp.total_spent.groupby(m).sum().reindex(range(1, 13), fill_value=0)
    sims.append((B + s).values)
sims = np.array(sims)
print(pd.Series(sims.argmin(axis=1) + 1).value_counts(normalize=True))  # bulan terendah
print(pd.Series(sims.argmax(axis=1) + 1).value_counts(normalize=True))  # bulan tertinggi

# 6. Cek bias
print(df.groupby('date_imputed').total_spent.mean())
print(df.groupby('date_imputed').item.value_counts(normalize=True).unstack())

460 4173.5
0.9435839319905743 0.9300699300699302
2    0.992
9    0.006
7    0.002
Name: proportion, dtype: float64
6     0.504
10    0.332
1     0.104
3     0.030
4     0.015
12    0.013
8     0.002
Name: proportion, dtype: float64
date_imputed
False    8.910769
True     9.072826
Name: total_spent, dtype: float64
item              Cake    Coffee    Cookie     Juice     Salad  Sandwich  \
date_imputed                                                               
False         0.138094  0.130649  0.120688  0.117857  0.127608  0.112824   
True          0.152174  0.097826  0.134783  0.102174  0.121739  0.121739   

item          Smoothie       Tea  
date_imputed                      
False         0.132536  0.119744  
True          0.128261  0.141304  
